# 🚀 Credit Card Dispute Resolver — Backend on Colab

**Before running anything:** Go to `Runtime → Change runtime type → T4 GPU`

Then run cells in order: **1 → 2 → 3 → 4**.
Cell 4 prints your public URL — copy it into `vite.config.js`

In [ ]:
# ── CELL 1 — Install dependencies ─────────────────────────────────────────────

!pip install -q --upgrade pip

!pip install -q \
fastapi \
"uvicorn[standard]" \
pydantic \
python-multipart \
python-dotenv \
torch \
transformers \
sentence-transformers \
scikit-learn==1.6.1 \
joblib \
numpy \
chromadb \
langchain \
langchain-core \
langchain-community \
langchain-huggingface \
langchain-chroma \
langchain-groq \
pyngrok

import torch
import numpy as np
import transformers
import sentence_transformers
import chromadb

print("✅ Dependencies installed")
print(f"numpy                  : {np.__version__}")
print(f"torch                  : {torch.__version__}")
print(f"transformers           : {transformers.__version__}")
print(f"sentence-transformers  : {sentence_transformers.__version__}")
print(f"chromadb               : {chromadb.__version__}")
print(f"GPU available          : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name               : {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
✅ Dependencies installed
numpy                  : 2.0.2
torch                  : 2.11.0+cu128
transformers           : 5.13.1
sentence-transformers  : 5.6.0
chromadb               : 1.5.9
GPU available          : True
GPU name               : Tesla T4


In [ ]:
# ── CELL 2 — Mount Google Drive & verify model files ──────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# ⚠️  Change this if your folder has a different name in Google Drive
MODEL_BASE = '/content/drive/MyDrive/dispute_model'
os.environ["MODEL_BASE_PATH"] = "/content/drive/MyDrive/dispute_model"

REQUIRED = [
    'bert_dispute_model/config.json',
    'bert_dispute_model/model.safetensors',
    'bert_dispute_model/tokenizer.json',
    'bert_dispute_model/tokenizer_config.json',
    'bert_dispute_model/vocab.txt',
    'bert_dispute_model/label_encoder.pkl',
    'baseline_lr_model.pkl',
    'tfidf_vectorizer.pkl',
    'label_encoder.pkl',
]

print(f'Checking: {MODEL_BASE}')
print('='*55)
all_ok = True
for rel in REQUIRED:
    full = os.path.join(MODEL_BASE, rel)
    exists = os.path.exists(full)
    size = f'{os.path.getsize(full)/1024/1024:.1f} MB' if exists else 'MISSING'
    icon = '✅' if exists else '❌'
    print(f'  {icon}  {rel:<50} {size}')
    if not exists: all_ok = False

print('='*55)
print('✅ All files found — ready to launch!' if all_ok else '❌ Some files missing — upload them to Google Drive first.')

Mounted at /content/drive
Checking: /content/drive/MyDrive/dispute_model
  ✅  bert_dispute_model/config.json                     0.0 MB
  ✅  bert_dispute_model/model.safetensors               255.4 MB
  ✅  bert_dispute_model/tokenizer.json                  0.7 MB
  ✅  bert_dispute_model/tokenizer_config.json           0.0 MB
  ✅  bert_dispute_model/vocab.txt                       0.2 MB
  ✅  bert_dispute_model/label_encoder.pkl               0.0 MB
  ✅  baseline_lr_model.pkl                              0.9 MB
  ✅  tfidf_vectorizer.pkl                               0.8 MB
  ✅  label_encoder.pkl                                  0.0 MB
✅ All files found — ready to launch!


In [ ]:
# ── CELL 3 — Write backend files to Colab disk ────────────────────────────────

rag_engine_code = r"""
import os
import logging
from pathlib import Path
import traceback

import chromadb
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
CHROMA_DIR  = Path("/content/drive/MyDrive/dispute_model/chroma_db")
COLLECTION  = "policy_docs"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ── Resolution rules (instant — no LLM needed) ────────────────────────────────
RESOLUTION_RULES = {
    "Unauthorized Transaction": {
        "action":             "Initiate chargeback investigation immediately",
        "timeline":           "3–10 business days",
        "priority":           "HIGH",
        "provisional_credit": True,
        "required_docs": [
            "Government-issued ID",
            "Credit card statement highlighting the transaction",
            "Police report if amount exceeds $500",
        ],
    },
    "Billing Error": {
        "action":             "Request itemised billing statement from merchant",
        "timeline":           "5–7 business days",
        "priority":           "MEDIUM",
        "provisional_credit": False,
        "required_docs": [
            "Original invoice or receipt",
            "Credit card statement",
        ],
    },
    "Duplicate Charge": {
        "action":             "Verify with merchant and reverse the duplicate charge",
        "timeline":           "2–5 business days",
        "priority":           "MEDIUM",
        "provisional_credit": False,
        "required_docs": [
            "Credit card statement showing both identical charges",
        ],
    },
    "Goods Not Received": {
        "action":             "Contact merchant for delivery proof; escalate chargeback if no response in 5 days",
        "timeline":           "5–15 business days",
        "priority":           "MEDIUM",
        "provisional_credit": False,
        "required_docs": [
            "Order confirmation email",
            "Tracking number",
            "Communication records with merchant",
        ],
    },
    "Service Not Provided": {
        "action":             "Request cancellation confirmation; initiate service-not-rendered chargeback",
        "timeline":           "5–10 business days",
        "priority":           "MEDIUM",
        "provisional_credit": False,
        "required_docs": [
            "Cancellation confirmation email or reference number",
            "Statement showing continued charges after cancellation",
        ],
    },
    "Merchant Fraud": {
        "action":             "Escalate to fraud department; block merchant; initiate full chargeback",
        "timeline":           "3–7 business days",
        "priority":           "URGENT",
        "provisional_credit": True,
        "required_docs": [
            "Screenshots of fraudulent merchant website",
            "All communication records",
            "Transaction records",
        ],
    },
}

# ── Prompt ────────────────────────────────────────────────────────────────────
_PROMPT = ChatPromptTemplate.from_template('''
You are a professional bank dispute resolution officer.

A customer submitted this complaint:
"{complaint}"

Our NLP classifier predicted the dispute category as: {category}

Relevant bank policy (use this as your only reference):
{context}

Write a professional response in exactly 3 sentences:
1. Explain why this complaint falls under "{category}"
2. Describe what the resolution process involves
3. Tell the customer what to expect (timeline and likely outcome)

Be factual, concise, and professional. No bullet points or headers.
''')


# ── Engine class ──────────────────────────────────────────────────────────────
class PolicyRAG:
    def __init__(self):
        self._vectorstore = None
        self._llm         = None
        self._embeddings  = None
        self.ready        = False

    def load(self) -> None:
        groq_key = os.environ.get("GROQ_API_KEY", "")

        # 1. Embedding model (local, free, no API key)
        log.info("Loading embedding model...")
        self._embeddings = HuggingFaceEmbeddings(
            model_name=EMBED_MODEL,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )

        # 2. Load ChromaDB from disk — chromadb 1.x API
        if not CHROMA_DIR.exists():
            log.error(
                f"ChromaDB not found at {CHROMA_DIR}. "
                "Run notebook 05_rag_setup.ipynb first to build the index."
            )
            return

        log.info(f"Loading ChromaDB from {CHROMA_DIR}...")
        # chromadb 1.x: use PersistentClient, then wrap with langchain-chroma
        chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
        self._vectorstore = Chroma(
            client=chroma_client,
            collection_name=COLLECTION,
            embedding_function=self._embeddings,
        )
        count = self._vectorstore._collection.count()
        log.info(f"✅ ChromaDB loaded — {count} vectors")

        # 3. Connect Groq LLM
        if groq_key:
            self._llm = ChatGroq(
                model="llama-3.3-70b-versatile",
                temperature=0.2,
                max_tokens=512,
                api_key=groq_key,
            )
            log.info("✅ Groq LLM connected (llama-3.3-70b-versatile)")
        else:
            log.warning("GROQ_API_KEY not set — explanations will use rule-based fallback")

        self.ready = True

    # ── Internal helpers ──────────────────────────────────────────────────────
    def _retrieve(self, complaint: str, category: str) -> str:
        retriever = self._vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={
                "k": 3,
                "filter": {"category": category},
            },
        )
        docs = retriever.invoke(complaint)

        # Fallback: if category filter returns nothing, search without filter
        if not docs:
            retriever_nofilter = self._vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={"k": 3},
            )
            docs = retriever_nofilter.invoke(f"{category} {complaint}")

        return "\n\n".join(doc.page_content for doc in docs)

    def _generate(self, complaint: str, category: str, context: str) -> str:
        chain = _PROMPT | self._llm | StrOutputParser()
        return chain.invoke({
            "complaint": complaint,
            "category":  category,
            "context":   context,
        })

    def _fallback_explanation(self, category: str) -> str:
        r = RESOLUTION_RULES.get(category, {})
        action   = r.get("action", "review the complaint")
        timeline = r.get("timeline", "5–10 business days")
        return (
            f"This complaint has been classified as '{category}' based on the key "
            f"details and language provided by the customer. "
            f"Our team will {action.lower()} to address the issue promptly. "
            f"You can expect a resolution within {timeline}."
        )

    # ── Public API ────────────────────────────────────────────────────────────
    def explain(self, complaint: str, category: str) -> dict:
        if not self.ready:
            return {
                "summary":          self._fallback_explanation(category),
                "policy_reference": f"{category.lower().replace(' ', '_')}.md",
                "used_llm":         False,
            }

        context = self._retrieve(complaint, category)

        if self._llm and context:
            try:
                summary  = self._generate(complaint, category, context)
                used_llm = True
            except Exception as e:
                log.error(f"Groq generation failed: {e}")
                summary  = self._fallback_explanation(category)
                used_llm = False
                traceback.print_exc()
        else:
            summary  = self._fallback_explanation(category)
            used_llm = False

        return {
            "summary":          summary,
            "policy_reference": f"{category.lower().replace(' ', '_')}.md",
            "used_llm":         used_llm,
        }

    def get_resolution(self, category: str) -> dict:
        return RESOLUTION_RULES.get(category, {
            "action":             "Review complaint manually",
            "timeline":           "5–10 business days",
            "priority":           "MEDIUM",
            "provisional_credit": False,
            "required_docs":      ["Supporting documentation"],
        })
"""

severity_scorer_code = r"""
import re

CATEGORY_BASE = {
    "Merchant Fraud":           4,
    "Unauthorized Transaction": 3,
    "Goods Not Received":       2,
    "Service Not Provided":     2,
    "Duplicate Charge":         1,
    "Billing Error":            1,
}

def _max_dollar(text: str) -> float:
    amounts = re.findall(r'\$\s*[\d,]+(?:\.\d{1,2})?', text)
    return max((float(a.replace("$","").replace(",","")) for a in amounts), default=0.0)

def score_complaint(complaint_text: str, predicted_category: str) -> dict:
    text    = complaint_text.lower()
    score   = 0
    factors = []

    base = CATEGORY_BASE.get(predicted_category, 1)
    score += base
    factors.append(f"{predicted_category} (+{base})")

    amt = _max_dollar(complaint_text)
    if amt >= 1000:
        score += 3; factors.append(f"High amount ${amt:,.0f} (+3)")
    elif amt >= 200:
        score += 2; factors.append(f"Amount ${amt:,.0f} (+2)")
    elif amt > 0:
        score += 1; factors.append(f"Amount ${amt:,.0f} (+1)")

    if any(s in text for s in ["months", "weeks", "multiple times", "again",
                                 "still", "keep charging", "repeatedly",
                                 "ongoing", "never resolved", "several times"]):
        score += 1; factors.append("Ongoing issue (+1)")

    if any(s in text for s in ["identity theft", "police report", "social security",
                                 "account hacked", "data breach"]):
        score += 2; factors.append("Security risk (+2)")

    if any(s in text for s in ["urgent", "desperate", "cannot afford",
                                 "emergency", "devastating", "losing money"]):
        score += 1; factors.append("Distress signals (+1)")

    if any(s in text for s in ["elderly", "disabled", "fixed income",
                                 "retirement", "pension"]):
        score += 1; factors.append("Vulnerable customer (+1)")

    score = min(score, 10)

    if score >= 8:   level, color = "URGENT", "#9b2c2c"
    elif score >= 5: level, color = "HIGH",   "#e53e3e"
    elif score >= 3: level, color = "MEDIUM", "#ed8936"
    else:            level, color = "LOW",    "#48bb78"

    return {"score": score, "level": level, "color": color, "factors": factors}
"""

main_code = r"""
import os, re, logging, joblib
import numpy as np
from pathlib import Path
from contextlib import asynccontextmanager
import traceback

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

from rag_engine import PolicyRAG
from severity_scorer import score_complaint

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ── Model paths ───────────────────────────────────────────────────────────────
_DEFAULT_BASE = Path(__file__).parent / "models"

def get_model_base() -> Path:
    env = os.environ.get("MODEL_BASE_PATH")
    return Path(env) if env else _DEFAULT_BASE

# ── Global holders ────────────────────────────────────────────────────────────
class Models:
    bert_model = bert_tokenizer = bert_le = None
    lr_model = tfidf = lr_le = None
    device = None
    bert_loaded = lr_loaded = False

rag = PolicyRAG()

# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_for_bert(text: str) -> str:
    text = re.sub(r'\bXX/XX/(?:XXXX|\d{4})\b', 'DATE', text)
    text = re.sub(r'\bXX/XX\b', 'DATE', text)
    text = re.sub(r'\$[\d,]+(?:\.\d{1,2})?', 'AMOUNT', text)
    text = re.sub(r'\bXX+\b', '', text)
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_for_tfidf(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\bxx+\b', '', text)
    text = re.sub(r'\$[\d,]+\.?\d*', 'amount', text)
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# ── Lifespan ──────────────────────────────────────────────────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    base = get_model_base()
    Models.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Device: {Models.device}")

    # DistilBERT
    bert_path = base / "bert_dispute_model"
    if bert_path.exists():
        try:
            Models.bert_tokenizer = AutoTokenizer.from_pretrained(str(bert_path))
            Models.bert_model = AutoModelForSequenceClassification.from_pretrained(
                str(bert_path)).to(Models.device)
            Models.bert_model.eval()
            Models.bert_le = joblib.load(bert_path / "label_encoder.pkl")
            Models.bert_loaded = True
            log.info("✅ DistilBERT loaded")
        except Exception as e:
            log.error(f"DistilBERT failed: {e}")
            traceback.print_exc()

    # TF-IDF baseline
    lr_path = base / "baseline_lr_model.pkl"
    if lr_path.exists():
        try:
            Models.lr_model = joblib.load(lr_path)
            Models.tfidf    = joblib.load(base / "tfidf_vectorizer.pkl")
            Models.lr_le    = joblib.load(base / "label_encoder.pkl")
            Models.lr_loaded = True
            log.info("✅ Baseline loaded")
        except Exception as e:
            log.error(f"Baseline failed: {e}")
            traceback.print_exc()

    # RAG engine
    try:
        rag.load()
        log.info(f"✅ RAG engine loaded (ready={rag.ready})")
    except Exception as e:
        log.error(f"RAG load failed: {e}")
        traceback.print_exc()

    yield
    log.info("Shutdown complete.")

# ── App ───────────────────────────────────────────────────────────────────────
app = FastAPI(
    title="Credit Card Dispute Resolver API",
    description="DistilBERT (89.98%) + LangChain RAG (Groq llama-3.3-70b) + Severity Scoring",
    version="2.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Schemas ───────────────────────────────────────────────────────────────────
class PredictRequest(BaseModel):
    complaint_text: str = Field(..., min_length=10, max_length=5000,
        example="I noticed a $299 charge I never made on my statement.")
    model: str = Field(default="bert")

class FullPredictRequest(BaseModel):
    complaint_text: str = Field(..., min_length=10, max_length=5000,
        example="I noticed a $299 charge I never made on my statement.")
    model: str          = Field(default="bert")
    include_rag:        bool = Field(default=True)
    include_severity:   bool = Field(default=True)

class CategoryScore(BaseModel):
    category: str
    score: float

class ClassificationResult(BaseModel):
    predicted_category: str
    confidence: float
    all_scores: list[CategoryScore]
    model_used: str
    text_preview: str

class SeverityResult(BaseModel):
    score: int
    level: str
    color: str
    factors: list[str]

class RAGResult(BaseModel):
    summary: str
    policy_reference: str
    used_llm: bool

class ResolutionResult(BaseModel):
    action: str
    timeline: str
    priority: str
    provisional_credit: bool
    required_docs: list[str]

class FullPredictResponse(BaseModel):
    classification: ClassificationResult
    severity: SeverityResult | None = None
    rag_explanation: RAGResult | None = None
    resolution: ResolutionResult | None = None

# ── Inference ─────────────────────────────────────────────────────────────────
def _run_bert(text: str) -> tuple:
    cleaned = clean_for_bert(text)
    inputs = Models.bert_tokenizer(
        cleaned, return_tensors="pt", truncation=True, max_length=256, padding=True)
    inputs = {k: v.to(Models.device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(
            Models.bert_model(**inputs).logits, dim=1).cpu().numpy()[0]
    idx   = int(np.argmax(probs))
    label = Models.bert_le.inverse_transform([idx])[0]
    scores = [CategoryScore(
        category=Models.bert_le.inverse_transform([i])[0],
        score=float(p)) for i, p in enumerate(probs)]
    return label, float(probs[idx]), scores

def _run_baseline(text: str) -> tuple:
    cleaned = clean_for_tfidf(text)
    probs   = Models.lr_model.predict_proba(Models.tfidf.transform([cleaned]))[0]
    idx     = int(np.argmax(probs))
    label   = Models.lr_le.inverse_transform([idx])[0]
    scores  = [CategoryScore(
        category=Models.lr_le.inverse_transform([i])[0],
        score=float(p)) for i, p in enumerate(probs)]
    return label, float(probs[idx]), scores

def _classify(complaint_text: str, model_choice: str) -> ClassificationResult:
    choice = model_choice.lower().strip()
    if choice == "bert":
        if not Models.bert_loaded:
            raise HTTPException(503, "DistilBERT not loaded. Use model='baseline'.")
        cat, conf, scores = _run_bert(complaint_text)
        used = "DistilBERT (fine-tuned · 89.98%)"
    elif choice == "baseline":
        if not Models.lr_loaded:
            raise HTTPException(503, "Baseline model not loaded.")
        cat, conf, scores = _run_baseline(complaint_text)
        used = "TF-IDF + Logistic Regression (82.44%)"
    else:
        raise HTTPException(400, f"Unknown model '{choice}'. Use 'bert' or 'baseline'.")

    return ClassificationResult(
        predicted_category=cat,
        confidence=conf,
        all_scores=sorted(scores, key=lambda x: -x.score),
        model_used=used,
        text_preview=complaint_text[:120] + ("..." if len(complaint_text) > 120 else ""),
    )

# ── Routes ────────────────────────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "message": "Credit Card Dispute Resolver API v2",
        "docs": "/docs",
        "endpoints": {
            "fast_classify":   "POST /predict",
            "full_analysis":   "POST /predict-full",
            "health":          "GET  /health",
            "models":          "GET  /models",
        }
    }

@app.get("/health")
def health():
    return {
        "status": "ok" if (Models.bert_loaded or Models.lr_loaded) else "degraded",
        "models_loaded": {
            "bert":     Models.bert_loaded,
            "baseline": Models.lr_loaded,
        },
        "rag_ready":    rag.ready,
        "cuda":         torch.cuda.is_available(),
        "device":       str(Models.device),
    }

@app.get("/models")
def list_models():
    return {"available": [
        {"id": "bert",     "name": "DistilBERT (fine-tuned)", "accuracy": "89.98%", "loaded": Models.bert_loaded, "recommended": True},
        {"id": "baseline", "name": "TF-IDF + LR",             "accuracy": "82.44%", "loaded": Models.lr_loaded,  "recommended": False},
    ]}

@app.post("/predict", response_model=ClassificationResult)
def predict(req: PredictRequest):
    result = _classify(req.complaint_text, req.model)
    log.info(f"/predict → {result.predicted_category} ({result.confidence:.2%})")
    return result

@app.post("/predict-full", response_model=FullPredictResponse)
def predict_full(req: FullPredictRequest):
    classification = _classify(req.complaint_text, req.model)
    category       = classification.predicted_category
    log.info(f"/predict-full → {category} ({classification.confidence:.2%})")

    severity = None
    if req.include_severity:
        raw = score_complaint(req.complaint_text, category)
        severity = SeverityResult(**raw)

    rag_explanation = None
    if req.include_rag and rag.ready:
        try:
            raw_rag = rag.explain(req.complaint_text, category)
            rag_explanation = RAGResult(**raw_rag)
        except Exception as e:
            log.error(f"RAG explain failed: {e}")
            traceback.print_exc()

    raw_res = rag.get_resolution(category)
    resolution = ResolutionResult(
        action             = raw_res.get("action", "Review manually"),
        timeline           = raw_res.get("timeline", "5–10 business days"),
        priority           = raw_res.get("priority", "MEDIUM"),
        provisional_credit = raw_res.get("provisional_credit", False),
        required_docs      = raw_res.get("required_docs", []),
    )

    return FullPredictResponse(
        classification=classification,
        severity=severity,
        rag_explanation=rag_explanation,
        resolution=resolution,
    )

@app.post("/batch-predict")
def batch_predict(requests: list[PredictRequest]):
    if len(requests) > 20:
        raise HTTPException(400, "Max 20 items per batch.")
    return [predict(r) for r in requests]
"""

with open('/content/rag_engine.py', 'w') as f:
    f.write(rag_engine_code)

with open('/content/severity_scorer.py', 'w') as f:
    f.write(severity_scorer_code)

with open('/content/main.py', 'w') as f:
    f.write(main_code)

print("✅ rag_engine.py  written to /content/")
print("✅ severity_scorer.py written to /content/")
print("✅ main.py written to /content/")


✅ rag_engine.py  written to /content/
✅ severity_scorer.py written to /content/
✅ main.py written to /content/


In [ ]:
# ── CELL 4 — Launch FastAPI + ngrok ───────────────────────────────────────────
import os
import traceback
from google.colab import userdata

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

import time, requests, asyncio, threading
import uvicorn
from pyngrok import ngrok, conf

def run_server():
    import asyncio
    import traceback
    import uvicorn

    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        config = uvicorn.Config(
            "main:app",
            host="0.0.0.0",
            port=8000,
            log_level="info",
            loop="asyncio",
        )

        server = uvicorn.Server(config)
        loop.run_until_complete(server.serve())

    except Exception:
        print("=" * 60)
        print("UVICORN STARTUP FAILED")
        print("=" * 60)
        traceback.print_exc()

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
time.sleep(1)

# Start server thread before ngrok so port 8000 is ready
t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(5)   # give uvicorn time to bind

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

with open('/content/public_url.txt', 'w') as f:
    f.write(PUBLIC_URL)

print('='*60)
print('  🚀 BACKEND IS LIVE!')
print('='*60)
print(f'  Public URL : {PUBLIC_URL}')
print(f'  API Docs   : {PUBLIC_URL}/docs')
print(f'  Health     : {PUBLIC_URL}/health')
print('='*60)
print()
print('  📋 Paste into frontend/vite.config.js:')
print(f'     target: "{PUBLIC_URL}"')
print()
print('  ⚠️  Do NOT stop this cell — stopping kills the server')
print()

# Self-test
time.sleep(3)
try:
    r = requests.get(f"{PUBLIC_URL}/health", timeout=100)

    print("HTTP status:", r.status_code)
    print("Response:")
    print(r.text)

    if r.ok:
        h = r.json()
        print(h)
    # h = requests.get(f'{PUBLIC_URL}/health', timeout=20).json()
    print('  Self-test results:')
    print(f'    status   : {h["status"]}')
    print(f'    bert     : {h["models_loaded"]["bert"]}')
    print(f'    baseline : {h["models_loaded"]["baseline"]}')
    print(f'    rag_ready: {h["rag_ready"]}')
    print(f'    cuda     : {h["cuda"]}')
    print()
    print('  ✅ Server is healthy!')
except Exception as e:
    print(f'  ⚠️  Health check failed: {e}')
    print('  Wait 15 seconds then try: requests.get(PUBLIC_URL + "/health").json()')
    traceback.print_exc()

# Keep cell alive
while True:
    time.sleep(300)
    print(f'  ⏱️  Still running — {PUBLIC_URL}')


INFO:     Started server process [1219]
INFO:     Waiting for application startup.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


  🚀 BACKEND IS LIVE!
  Public URL : https://afterlife-crudeness-spent.ngrok-free.dev
  API Docs   : https://afterlife-crudeness-spent.ngrok-free.dev/docs
  Health     : https://afterlife-crudeness-spent.ngrok-free.dev/health

  📋 Paste into frontend/vite.config.js:
     target: "https://afterlife-crudeness-spent.ngrok-free.dev"

  ⚠️  Do NOT stop this cell — stopping kills the server

INFO:     34.16.190.36:0 - "GET /health HTTP/1.1" 200 OK
HTTP status: 200
Response:
{"status":"ok","models_loaded":{"bert":true,"baseline":true},"rag_ready":true,"cuda":true,"device":"cuda"}
{'status': 'ok', 'models_loaded': {'bert': True, 'baseline': True}, 'rag_ready': True, 'cuda': True, 'device': 'cuda'}
  Self-test results:
    status   : ok
    bert     : True
    baseline : True
    rag_ready: True
    cuda     : True

  ✅ Server is healthy!
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1

In [ ]:
# ── CELL 5 — Test all 6 categories ────────────────────────────────────────────
# Run this in a SEPARATE tab after Cell 4 is running
import requests, json

# Reads the URL that Cell 4 saved automatically — no copy-pasting needed
with open('/content/public_url.txt') as f:
    BASE = f.read().strip()

print(f'  Testing against: {BASE}')

TESTS = [
    ('Unauthorized Transaction', 'Someone used my credit card without my permission and made a $500 purchase I never authorized.'),
    ('Duplicate Charge',         'I was charged twice for the same $89.99 purchase on the same day from the same merchant.'),
    ('Goods Not Received',       'I ordered a laptop three months ago and it never arrived. The seller stopped responding.'),
    ('Billing Error',            'I was overcharged. The merchant agreed to charge $45 but billed me $145. Clear billing error.'),
    ('Service Not Provided',     'I cancelled my gym membership in January but they kept charging my card every month after.'),
    ('Merchant Fraud',           'The website was fake. I paid $200 but the merchant never existed. It was a complete scam.'),
]

print('='*65)
print('  PREDICTION TESTS (DistilBERT)')
print('='*65)
passed = 0
for expected, text in TESTS:
    r = requests.post(f'{BASE}/predict', json={'complaint_text': text, 'model': 'bert'}, timeout=30)
    d = r.json()
    got  = d['predicted_category']
    conf = d['confidence'] * 100
    ok   = got == expected
    if ok: passed += 1
    icon = '✅' if ok else '❌'
    match = '(correct)' if ok else f'(expected: {expected})'
    print(f'  {icon}  {got:<30} {conf:5.1f}%  {match}')

print('='*65)
print(f'  {passed}/6 correct  —  {"Perfect! 🎉" if passed == 6 else "Check mismatches above."}')
